# 06 - Best Model Selection & Deployment Artifact (Hugging Face)

This notebook performs the **final model selection and deployment preparation** step.

Goals:

1. **Scan all trained models** saved in the **Gold layer** (`GOLD_PATH`) and identify the best model based on F1-score.  
2. **Align the chosen model** with the requirements of the Hugging Face app:  
   - Use a well-defined subset of features  
   - Rebuild a clean, explicit training pipeline  
   - Reuse the best hyperparameters discovered earlier  
3. **Retrain the final XGBoost pipeline** on the engineered dataset.  
4. **Optimize the decision threshold** for F1-score.  
5. **Export a deployment-ready `model.pkl`** using `ModelWrapper`, stored in `HUGGING_DIR`, to be consumed by the Gradio app.

This notebook closes the modeling loop: from multiple candidates to a single, production-ready model.

In [ ]:
# --- Basic Imports ---
from utils import *
import warnings
warnings.filterwarnings("ignore", message=".*serialized model.*")
import shutil

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score,
                             precision_recall_fscore_support, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, log_loss)

from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from category_encoders import WOEEncoder
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier, Booster
import xgboost as xgb
import shap
import json


In [ ]:
def find_best_threshold(y_true, y_prob, metric="f1"):
    thresholds = np.linspace(0.01, 0.99, 200)
    scores = []

    for t in thresholds:
        y_pred = (y_prob > t).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )

        if metric == "precision":
            scores.append(precision)
        elif metric == "recall":
            scores.append(recall)
        else:  # default f1
            scores.append(f1)

    scores = np.array(scores)
    best_idx = scores.argmax()
    best_threshold = thresholds[best_idx]
    best_score = scores[best_idx]

    return best_threshold, best_score, thresholds, scores, metric

def replace_unknowns(df):
    return df.replace("unknown", np.nan)

## 1. Scan Gold Models & Select the Best Candidate

We begin by scanning all models saved in the **Gold layer** (`GOLD_PATH`) and loading each `ModelWrapper`:

- Each model stores key metrics and metadata (including F1-score) in `model.metadata`.
- We loop through all `.pkl` files, load their wrappers, and track the **best F1-score**.
- The model with the **highest F1** is selected as the best candidate.

This gives us an automatic, metric-driven way to identify the most promising model across:

- Baseline Logistic Regression  
- Tuned Logistic Regression  
- Tuned XGBoost  
- Any other variants persisted during experimentation

In [ ]:
models = os.listdir(GOLD_PATH)
models = [f for f in models if f.endswith('.pkl')]

best_model_file = None
best_f1 = -1

for model_file in models:
    try:
        model_path = os.path.join(GOLD_PATH, model_file)
        # print(f"Loading model from: {model_path}")
        model = ModelWrapper.load(model_path)
        f1_score = model.metadata['F1']
        # print(f"Evaluating model: {model_file}, F1: {f1_score}")
        if f1_score >= best_f1:
            best_f1 = f1_score
            best_model_file = model_file
    except Exception as e:
        print(f"Failed to load/evaluate model {model_file}: {e}")

print(f"Best model: {best_model_file} with F1: {best_f1}")

## 2. Final Model Choice for Deployment (XGBoost)

Although we can automatically identify the best model by F1-score, we also want to ensure:

- **Consistency with the Gradio interface** (expected features, data schema)
- **Stability and reproducibility** (choosing a specific, known-good run)
- **Alignment with the project’s narrative** (XGBoost as the final model)

For this reason, we **explicitly select** a particular XGBoost model file from the Gold layer, corresponding to a run we have validated: XGBoost_Model_YYYYMMDD_HHMMSS.pkl

In [ ]:
# From best_model_file, extract date and time, convert to yyyy-MM-mm hh:mm:ss format
timestamp = best_model_file.split('_')[2] + '_' + best_model_file.split('_')[3].split('.')[0]
timestamp = dt.datetime.strptime(timestamp, "%Y%m%d_%H%M%S")
timestamp_str = timestamp.strftime("%Y-%m-%d %H:%M:%S")

best_model_file = 'XGBoost_Model_20251205_154421.pkl'

best_model_path = os.path.join(GOLD_PATH, 'XGBoost_Model_20251205_154421.pkl')
model = ModelWrapper.load(best_model_path)

### 2.1 Align Feature Names & Hyperparameters

From the selected `XGBoost_Model_*.pkl`, we:

- Extract the **final feature names** used by the model  
- Clean them by:
  - Removing technical prefixes like `num__`
  - Dropping PCA component names (we will recompute PCA in a fresh pipeline)
- Append back the original macroeconomic variables (`emp_var_rate`, `euribor3m`, `nr_employed`, `cons_price_idx`) so we can rebuild the preprocessing from scratch.

Next, we:

- Load the corresponding row from `XGBoost_metrics.csv` (matching the model timestamp)
- Parse the stored `Best Parameters` JSON into a dictionary
- Remove any `select__k` entries, because in this final deployment pipeline we use a slightly simplified setup (no feature selection step inside the ColumnTransformer)

These hyperparameters are then injected into a **clean XGBoost + SMOTE pipeline** trained on the engineered dataset.

In [ ]:
# Get feature names from the best model, remove num__ prefix and drop features that end with __PCA0 or 1 suffix
feature_names = [feat.replace('num__', '') for feat in model.metadata['Feature Names'] if not (feat.endswith('__pca0') or feat.endswith('__pca1'))] + ['emp_var_rate','euribor3m','nr_employed','cons_price_idx']

# From best_model_file, extract date and time, convert to yyyy-MM-mm hh:mm:ss format
timestamp = best_model_file.split('_')[2] + '_' + best_model_file.split('_')[3].split('.')[0]
timestamp = dt.datetime.strptime(timestamp, "%Y%m%d_%H%M%S")
timestamp_str = timestamp.strftime("%Y-%m-%d %H:%M:%S")

# Load the XGBoost.csv file for the timestamp identified
df = pd.read_csv(os.path.join(MODEL_DIR, 'XGBoost_Model/XGBoost_metrics.csv'), sep=";")
df = df[df['Timestamp'] == timestamp_str]
hyper_params = df.iloc[0]['Best Parameters']

# Convert hyper_params string to dictionary
hyper_params = json.loads(hyper_params.replace("'", '"'))

# Remove from hyper_params any keys that contain select__k
hyper_params = {k: v for k, v in hyper_params.items() if 'select__k' not in k}


## 3. Rebuild Training Dataset for the Deployment Model

We now reload the **XGBoost-engineered** Silver dataset: SILVER_PATH/base_dataset_XGB_Engineered.csv

In [ ]:
df = pd.read_csv(os.path.join(SILVER_PATH, 'base_dataset_XGB_Engineered.csv'), sep=';')

X = df[feature_names].copy()
y = df['y'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

macro_cols = ['emp_var_rate','euribor3m','nr_employed','cons_price_idx']

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_features = [col for col in numeric_features if col not in macro_cols]
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

## 4. Final XGBoost Deployment Pipeline

We reconstruct a **clean and explicit pipeline** for deployment:

1. **Cleaning**
   - `replace_unknowns`: converts `"unknown"` values to `NaN` so they can be handled by imputers

2. **Preprocessing (`ColumnTransformer`)**
   - *Macro block (`macro_economics`)*
     - Mean imputation → StandardScaler → PCA (2 components)
   - *Numeric block (`num`)*
     - Median imputation → StandardScaler
   - *Categorical block (`cat`)*
     - Most frequent imputation → WOE encoding

3. **SMOTE Resampling**
   - Use **SMOTE** with `k_neighbors` and `sampling_strategy` taken from the previously tuned hyperparameters  
   - This addresses class imbalance by creating synthetic minority-class samples during training.

4. **XGBoost Model**
   - `XGBClassifier` configured with the **exact hyperparameters** retrieved from the prior grid search:
     - `max_depth`, `learning_rate`, `n_estimators`, `subsample`, `colsample_bytree`, `min_child_weight`, etc.
   - Objective: `binary:logistic`, with `logloss` as evaluation metric  
   - `tree_method="hist"` for efficient training

We then fit this pipeline on the training set, resulting in a **production-ready model** consistent with our best-performing experiment.


In [ ]:
# Preprocessing pipelines for numeric and categorical data
pca_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))
])

numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("woe", WOEEncoder(handle_missing="value", handle_unknown="value"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("macro_economics", pca_pipeline, macro_cols),
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop"
)

cleaner = FunctionTransformer(replace_unknowns, validate=False, feature_names_out="one-to-one")

# SMOTE approach with hyper_params dictionary
clf = Pipeline(steps=[
    ("clean", cleaner),
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42,
                    k_neighbors=hyper_params["smote__k_neighbors"],
                    sampling_strategy=hyper_params["smote__sampling_strategy"])),
    ("model", XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        colsample_bytree=hyper_params["model__colsample_bytree"],
        learning_rate=hyper_params["model__learning_rate"],
        max_depth=hyper_params["model__max_depth"],
        min_child_weight=hyper_params["model__min_child_weight"],
        n_estimators=hyper_params["model__n_estimators"],
        subsample=hyper_params["model__subsample"],
        random_state=42
    ))
])

clf.fit(X_train, y_train)

## 5. Threshold Optimization & Final Evaluation

After training, we evaluate the model on the test set:

1. Compute predicted probabilities for the positive class (`y = 1`)  
2. Use `find_best_threshold()` to:
   - Sweep thresholds between 0.01 and 0.99  
   - Compute precision, recall, and F1 at each threshold  
   - Identify the threshold that **maximizes F1-score**

3. Apply the best threshold to obtain binary predictions (`y_pred_opt`)  
4. Print:
   - **Classification report** (precision, recall, F1 for each class)  
   - **Confusion matrix**, showing:
     - True negatives (correctly identified non-subscribers)
     - False positives (extra calls to non-subscribers)
     - False negatives (missed subscribers)
     - True positives (correctly identified subscribers)

This evaluation validates that the deployment model is consistent with the tuned XGBoost behavior observed in Notebook 05.

In [ ]:
y_prob = clf.predict_proba(X_test)[:, 1]

best_threshold, best_score, thresholds, scores, metric = find_best_threshold(
    y_test, y_prob, metric="f1"
)

# Apply best threshold
y_pred_opt = (y_prob > best_threshold).astype(int)

print("\nClassification Report with Best Threshold:")
print(classification_report(y_test, y_pred_opt))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_opt))

## 6. Export Deployment Model for Hugging Face

Finally, we wrap the trained pipeline and decision threshold in a `ModelWrapper`:

- `pipeline` → full preprocessing + SMOTE + XGBoost model  
- `threshold` → optimal probability threshold found via F1 optimization  

We then save this object as: HUGGING_DIR/model.pkl

In [ ]:
model = ModelWrapper(
    pipeline=clf,
    threshold=best_threshold,
)

model.save(HUGGING_DIR + "model.pkl")